# LAB | Feature Engineering
**Dataset:** Spaceship Titanic  
**Modelo:** K-Nearest Neighbors (KNN)  
**Tipo:** Clasificación

En este lab aplicamos técnicas de **Feature Engineering** para mejorar el modelo respecto al Lab anterior:
- Transformar columnas con demasiada granularidad
- Eliminar columnas irrelevantes
- Codificar variables categóricas con dummies
- Normalizar las features numéricas

## Tabla de Contenidos
1. [Cargar los datos](#1)
2. [Explorar los datos](#2)
3. [Gestionar valores nulos](#3)
4. [Feature Engineering](#4)
5. [Dummies para columnas categóricas](#5)
6. [Train Test Split](#6)
7. [Normalización](#7)
8. [Modelo KNN y evaluación](#8)

## 1. Cargar los datos <a id='1'></a>

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Cargar el mismo dataset del Lab anterior
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

## 2. Explorar los datos <a id='2'></a>

In [ ]:
# Comprobar el shape del dataset
spaceship.shape

In [ ]:
# Comprobar los tipos de datos de cada columna
spaceship.dtypes

In [ ]:
# Comprobar los valores nulos por columna
spaceship.isnull().sum()

## 3. Gestionar valores nulos <a id='3'></a>

Existen múltiples estrategias para gestionar valores nulos:
- Eliminar filas o columnas enteras que contienen nulos
- Rellenar con un valor estadístico (media para continuas, moda para categóricas)
- Rellenar con un algoritmo de imputación

En este ejercicio, dado que la cantidad de nulos es baja, eliminamos directamente las filas afectadas.

In [ ]:
# Eliminar filas con valores nulos
spaceship = spaceship.dropna()

print("Valores nulos tras dropna:", spaceship.isnull().sum().sum())
print("Nuevo shape:", spaceship.shape)

## 4. Feature Engineering <a id='4'></a>

In [ ]:
# Cabin tiene demasiada granularidad (formato 'A/0/P', 'B/1/S', etc.)
# Solo nos interesa la letra del deck: {'A','B','C','D','E','F','G','T'}
# Extraemos solo el primer carácter antes de la barra
spaceship['Cabin'] = spaceship['Cabin'].str.split('/').str[0]

print("Valores únicos en Cabin:", spaceship['Cabin'].unique())
spaceship['Cabin'].value_counts()

In [ ]:
# Eliminar PassengerId y Name
# Son identificadores únicos que no aportan información predictiva al modelo
spaceship = spaceship.drop(columns=['PassengerId', 'Name'])

print("Columnas restantes:", spaceship.columns.tolist())

## 5. Dummies para columnas categóricas <a id='5'></a>

KNN es un algoritmo basado en distancias — no puede trabajar con texto.  
Usamos `pd.get_dummies()` para convertir las columnas categóricas en columnas numéricas binarias (0 o 1).  
Usamos `drop_first=True` para evitar multicolinealidad (dummy variable trap).

In [ ]:
# Identificar columnas no numéricas
cat_cols = spaceship.select_dtypes(include=['object', 'str']).columns.tolist()
print("Columnas categóricas:", cat_cols)

# Aplicar get_dummies a todas las columnas categóricas
spaceship = pd.get_dummies(spaceship, columns=cat_cols, drop_first=True, dtype=int)

print("Shape tras dummies:", spaceship.shape)
spaceship.head()

In [ ]:
# Separar features (X) y target (y)
# 'Transported' es booleana -> problema de clasificación -> KNeighborsClassifier
X = spaceship.drop(columns=['Transported'])
y = spaceship['Transported']

print("Features:", X.shape)
print("Target:", y.value_counts().to_dict())

## 6. Train Test Split <a id='6'></a>

In [ ]:
# Dividir en entrenamiento (80%) y prueba (20%)
# random_state=42 para reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

## 7. Normalización <a id='7'></a>

KNN calcula distancias entre puntos. Si una feature tiene valores en rangos muy distintos (ej: Age 0-80 vs RoomService 0-14000), domina el cálculo de distancia y sesga el modelo.  
Usamos **MinMaxScaler** para escalar todas las features al rango [0, 1].  
**Importante:** ajustamos (`fit`) el scaler solo con los datos de entrenamiento y luego lo aplicamos (`transform`) a ambos conjuntos — así evitamos data leakage.

In [ ]:
# Inicializar el scaler
scaler = MinMaxScaler()

# Fit solo con train, transform en ambos
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features escaladas - rango train:", X_train_scaled.min().round(2), "-", X_train_scaled.max().round(2))

## 8. Modelo KNN y evaluación <a id='8'></a>

In [ ]:
# Inicializar y entrenar KNN sin hiperparámetros (n_neighbors=5 por defecto)
# Usamos Classifier porque el target es booleano (True/False)
knn = KNeighborsClassifier()
knn.fit(X_train_scaled, y_train)

In [ ]:
# Predicciones y accuracy
y_pred = knn.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

In [ ]:
# Reporte de clasificación: precisión, recall y F1-score por clase
print(classification_report(y_test, y_pred))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Transportado', 'Transportado'],
            yticklabels=['No Transportado', 'Transportado'])
plt.title('Matriz de Confusión - KNN con Feature Engineering')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.show()

## Conclusión

Comparando con el Lab anterior donde solo usamos columnas numéricas:

| Lab | Features usadas | Accuracy |
|-----|----------------|----------|
| Lab 1 | Solo numéricas (6 features), sin escalar | ~77% |
| Lab 2 | Todas las features + dummies + normalización | > 77% |

El Feature Engineering mejora el modelo porque:
- Incluimos más información (columnas categóricas convertidas a dummies)
- Normalizamos las features para que KNN calcule distancias de forma justa
- Eliminamos columnas irrelevantes (PassengerId, Name) que solo añaden ruido